In [ ]:
import json, gzip, pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── load SNMP data ────────────────────────────────────────────────────────────
BASE = pathlib.Path('T4')

def load_metric(name):
    with gzip.open(BASE / 'export_pi5_127.0.0.1.jsonl.gz', 'rt') as f:
        for line in f:
            d = json.loads(line)
            if d['metric'].get('__name__') == name:
                df = pd.DataFrame({
                    'time': pd.to_datetime([t/1000 for t in d['timestamps']], unit='s', utc=True),
                    'mbps': [v/1e6 for v in d['values']]
                }).set_index('time')
                return df

in_bps  = load_metric('snmp_in_bps')
out_bps = load_metric('snmp_out_bps')

# ── load CSV event logs ───────────────────────────────────────────────────────
def load_csv(fname):
    df = pd.read_csv(BASE / fname)
    df['time'] = pd.to_datetime(df['time_utc'], utc=True)
    df['iperf_tx_mbps'] = pd.to_numeric(df['iperf_tx_mbps'], errors='coerce')
    df['iperf_rx_mbps'] = pd.to_numeric(df['iperf_rx_mbps'], errors='coerce')
    return df

ev1 = load_csv('test4_snmp_validation_20261023T1505Z.csv')
ev2 = load_csv('test4_snmp_validation_20261023T1521Z.csv')

def t(ev, phase):
    return ev.loc[ev['phase'] == phase, 'time'].iloc[0]

# ── phase windows: (label, t_start, t_end, color, iperf_val, iperf_direction) ─
phases = [
    # Session 1 ---------------------------------------------------------------
    ('Baseline',        t(ev1,'run1_ph0_start'), t(ev1,'run1_ph0_end'),   '#cfe2ff', None,  None),
    ('Upload\n17.5 Mb/s', t(ev1,'run1_ph1_start'), t(ev1,'run1_ph1_end'), '#ffe0b2', 17.50, 'out'),
    ('Settle',          t(ev1,'run1_ph2_start'), t(ev1,'run1_ph2_end'),   '#e8f5e9', None,  None),
    ('Download\n(failed)', t(ev1,'run1_ph3_start'), t(ev1,'run1_ph3_end'),'#fce4ec', None,  None),
    ('Baseline',        t(ev1,'run1_ph4_start'), t(ev1,'run1_ph4_end'),   '#cfe2ff', None,  None),
    ('Baseline',        t(ev1,'run2_ph0_start'), t(ev1,'run2_ph0_end'),   '#cfe2ff', None,  None),
    ('Upload\n(failed)', t(ev1,'run2_ph1_start'), t(ev1,'run2_ph1_end'),  '#ffe0b2', None,  None),
    ('Settle',          t(ev1,'run2_ph2_start'), t(ev1,'run2_ph2_end'),   '#e8f5e9', None,  None),
    ('Download\n24.7 Mb/s', t(ev1,'run2_ph3_start'), t(ev1,'run2_ph3_end'),'#fce4ec', 24.69,'in'),
    ('Baseline',        t(ev1,'run2_ph4_start'), t(ev1,'run2_ph4_end'),   '#cfe2ff', None,  None),
    # Session 2 ---------------------------------------------------------------
    ('Baseline',        t(ev2,'run1_ph0_start'), t(ev2,'run1_ph0_end'),   '#cfe2ff', None,  None),
    ('Upload\n(failed)', t(ev2,'run1_ph1_start'), t(ev2,'run1_ph1_end'),  '#ffe0b2', None,  None),
    ('Settle',          t(ev2,'run1_ph2_start'), t(ev2,'run1_ph2_end'),   '#e8f5e9', None,  None),
    ('Download\n24.1 Mb/s', t(ev2,'run1_ph3_start'), t(ev2,'run1_ph3_end'),'#fce4ec', 24.07,'in'),
    ('Baseline',        t(ev2,'run1_ph4_start'), t(ev2,'run1_ph4_end'),   '#cfe2ff', None,  None),
    ('Baseline',        t(ev2,'run2_ph0_start'), t(ev2,'run2_ph0_end'),   '#cfe2ff', None,  None),
    ('Upload\n(failed)', t(ev2,'run2_ph1_start'), t(ev2,'run2_ph1_end'),  '#ffe0b2', None,  None),
    ('Settle',          t(ev2,'run2_ph2_start'), t(ev2,'run2_ph2_end'),   '#e8f5e9', None,  None),
    ('Download\n(failed)', t(ev2,'run2_ph3_start'), t(ev2,'run2_ph3_end'),'#fce4ec', None,  None),
    ('Baseline',        t(ev2,'run2_ph4_start'), t(ev2,'run2_ph4_end'),   '#cfe2ff', None,  None),
]

# ── PLOT ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))

# SNMP lines
ax.plot(out_bps.index, out_bps['mbps'], color='#e65100', lw=1.5, zorder=3,
        label='SNMP out_bps — Pi→router→LTE (upload traffic)')
ax.plot(in_bps.index,  in_bps['mbps'],  color='#1565c0', lw=1.5, zorder=3,
        label='SNMP in_bps  — LTE→router→Pi (download traffic)')

# phase shading + labels
label_done = set()
for (label, t0, t1, color, iperf_val, iperf_dir) in phases:
    ax.axvspan(t0, t1, color=color, alpha=0.55, zorder=1)
    mid = t0 + (t1 - t0) / 2
    ax.text(mid, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 30,
            label, ha='center', va='top', fontsize=7, color='#333333',
            transform=ax.get_xaxis_transform(), clip_on=True)

    # dashed iperf3 reference line
    if iperf_val:
        lc = '#e65100' if iperf_dir == 'out' else '#1565c0'
        ax.hlines(iperf_val, t0, t1, colors=lc, linestyles='--', lw=2, zorder=4)
        ax.text(t1, iperf_val + 0.4, f'iperf3={iperf_val:.1f}', fontsize=8,
                color=lc, va='bottom', ha='right')

# session separator
sep = t(ev2, 'run1_ph0_start')
ax.axvline(sep, color='black', lw=1.2, ls=':', zorder=5)
ax.text(sep, 0.98, ' Session 2 starts', transform=ax.get_xaxis_transform(),
        fontsize=8, va='top', color='black')

# session labels at top
s1_mid = t(ev1,'run1_ph0_start') + (t(ev1,'run2_ph4_end') - t(ev1,'run1_ph0_start'))/2
s2_mid = t(ev2,'run1_ph0_start') + (t(ev2,'run2_ph4_end') - t(ev2,'run1_ph0_start'))/2
ax.text(s1_mid, 1.01, 'Session 1  (15:05–15:20 UTC)', transform=ax.get_xaxis_transform(),
        ha='center', fontsize=9, fontweight='bold')
ax.text(s2_mid, 1.01, 'Session 2  (15:21–15:34 UTC)', transform=ax.get_xaxis_transform(),
        ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('Throughput (Mbit/s)', fontsize=11)
ax.set_xlabel('Time (UTC)', fontsize=11)
ax.set_title(
    'Test 4 – SNMP Validation: Router lte0 Counters vs iperf3 Ground Truth\n'
    'Dashed line = iperf3-reported rate  |  Solid line = SNMP-measured rate  |  '
    'Mean agreement: 100.4%',
    fontsize=11
)

legend_elements = [
    Line2D([0],[0], color='#e65100', lw=2, label='SNMP out_bps (upload)'),
    Line2D([0],[0], color='#1565c0', lw=2, label='SNMP in_bps (download)'),
    Line2D([0],[0], color='gray',    lw=2, ls='--', label='iperf3 ground truth'),
    mpatches.Patch(color='#cfe2ff', alpha=0.7, label='Baseline (idle)'),
    mpatches.Patch(color='#ffe0b2', alpha=0.7, label='Upload phase'),
    mpatches.Patch(color='#fce4ec', alpha=0.7, label='Download phase'),
    mpatches.Patch(color='#e8f5e9', alpha=0.7, label='Settle'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.25)
ax.set_ylim(bottom=0)

import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz='UTC'))

plt.tight_layout()
plt.savefig('t4_snmp_validation.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved t4_snmp_validation.png')

# ── quick numbers ─────────────────────────────────────────────────────────────
print('\n--- Valid iperf3 measurements ---')
valid = [
    ('S1 R1 upload',   17.50, 'out', t(ev1,'run1_ph1_start'), t(ev1,'run1_ph1_end')),
    ('S1 R2 download', 24.69, 'in',  t(ev1,'run2_ph3_start'), t(ev1,'run2_ph3_end')),
    ('S2 R1 download', 24.07, 'in',  t(ev2,'run1_ph3_start'), t(ev2,'run1_ph3_end')),
]
for label, iperf, direction, t0, t1 in valid:
    series = out_bps if direction == 'out' else in_bps
    w = series.loc[(series.index >= t0) & (series.index <= t1), 'mbps']
    print(f'{label:<20}  iperf3={iperf:.2f} Mbit/s  SNMP mean={w.mean():.2f}  ratio={w.mean()/iperf*100:.1f}%')
